In [ ]:
import base64
import requests
from ultralytics import YOLO
from dao.BaseDao import BaseDao
import os
import cv2
import numpy as np
import shutil


In [ ]:
def hsv_get(image_path):
    """
    再次分割书脊获得书标区域
    """
    # 读取图像
    image = cv2.imread(image_path)

    # 将图像转换为HSV颜色空间
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # 定义红色的HSV范围
    lower_red = np.array([0, 100, 100])    # 红色的低阈值
    upper_red = np.array([10, 255, 255])   # 红色的高阈值

    # 创建一个mask，其中红色区域为白色，其他区域为黑色
    mask = cv2.inRange(hsv, lower_red, upper_red)

    # 寻找红线区域的轮廓
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 设置上下偏移量
    y_offset_top = -3
    y_offset_bottom = 17

    # 在原始图像上绘制红线区域的轮廓（仅作为示例）
    if contours:
        # 对轮廓按面积排序，取最大的两个轮廓
        contours = sorted(contours, key=cv2.contourArea, reverse=True)[:2]

        # 获取两条红线的 y 坐标
        y_coords = []
        for contour in contours:
            _, y, _, _ = cv2.boundingRect(contour)
            y_coords.append(y)

        # 确定上下两条红线的 y 坐标并应用偏移量
        y_coords.sort()
        y_top = max(y_coords[0] - y_offset_top, 0)
        y_bottom = min(y_coords[1] + y_offset_bottom, image.shape[0])

        # 提取两条红线之间的区域
        between_region = image[y_top:y_bottom, :]

        # 保存结果
        shubiao = between_region
        # cv2.imwrite('output/hsv.jpg', between_region)

        print("两条最长红线之间的区域分割完成并保存")
    else:
        print("未找到红线区域，请调整阈值或检查图像")

In [ ]:
hsv_get("shuji.jpg")

In [ ]:
import os
results_with_question = dict()  # 存储存在疑问的结果
results_with_error = dict()  # 存储存在错误的结果
all_book_result_in_dict = dict()  # 存储所有书籍识别结果
# 清理旧的运行目录
if os.path.exists('./runs'):
    shutil.rmtree('./runs')
# 执行预处理步骤
dict_coordinate_data, sorted_coordinate_dicts, removed_id_dicts = preProcess()

# 请求并保存识别结果
print('确认请求 ...')

for name, sorted_dict in sorted_coordinate_dicts.items():
    all_book_result_in_list_dict = dict()  # 存储单个文件内所有书籍识别结果
    for key, xywh in sorted_dict:
        if key in removed_id_dicts[name]:  # 若ID已被移除，则跳过
            continue
        path = seq_to_filepath(name, key, result_dir)   # 构建文件路径  
        # 整合hsv_get函数
        hsv_get(path)
        hsv_path = 'out/between_regions_hsv.jpg'  # 由hsv_get函数保存的图像路径
        if os.path.exists(path):
            all_book_result_in_list_dict[key] = send_post_request(
                image_to_base64(path)
            ).json()  # 发送请求并获取响应JSON  存储识别结果
            # print('finish ' + str(key))
    all_book_result_in_dict[name] = all_book_result_in_list_dict  # 将单个文件的识别结果加入总结果字典

In [ ]:
def is_subsequence(a, b):
    """判断a是否是b的子序列"""
    sub_iter = iter(a)
    return all(char in sub_iter for char in b if char in b)

In [ ]:
is_subsequence("开始写吧", "开始")

In [ ]:
import cv2
import numpy as np

from utils.ImageExecute import *
# from utils.OCR import send_post_request
# from utils.ImageExecute import image_to_base64

def hsv_get(image_path):
    """
    再次分割书脊获得书标区域
    """
    # 读取图像
    image = cv2.imread(image_path)

    # 将图像转换为HSV颜色空间
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # 定义红色的HSV范围
    lower_red = np.array([0, 100, 100])    # 红色的低阈值
    upper_red = np.array([10, 255, 255])   # 红色的高阈值

    # 创建一个mask，其中红色区域为白色，其他区域为黑色
    mask = cv2.inRange(hsv, lower_red, upper_red)

    # 寻找红线区域的轮廓
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 设置上下偏移量
    y_offset_top = -22
    y_offset_bottom = 3

    # 在原始图像上绘制红线区域的轮廓（仅作为示例）
    if contours:
        # 对轮廓按面积排序，取最大的两个轮廓
        contours = sorted(contours, key=cv2.contourArea, reverse=True)[:2]

        # 获取两条红线的 y 坐标
        y_coords = []
        for contour in contours:
            _, y, _, _ = cv2.boundingRect(contour)
            y_coords.append(y)

        # 确定上下两条红线的 y 坐标并应用偏移量
        y_coords.sort()
        y_top = max(y_coords[0] - y_offset_top, 0)
        y_bottom = min(y_coords[1] + y_offset_bottom, image.shape[0])

        # 提取两条红线之间的区域
        between_region = image[y_top:y_bottom, :]

        #分割图片路径获得图片名称
        lujing = image_path.split('/')
        name = lujing[len(lujing)-1]

        # 保存结果
        cv2.imwrite('./hsv/' + name, between_region)
        # shubiao = between_region

        # print("两条最长红线之间的区域分割完成并保存")
        print('./hsv/' + name)
        return './hsv/' + name
    else:
        print("未找到红线区域，请调整阈值或检查图像")
        return 0
    
def preprocess_image(image):
    """
    对图像进行预处理，包括对比度增强、二值化和去噪。
    """
    # 增强对比度
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    image = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    # 转为灰度图
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 二值化
    _, binary_image = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 去噪
    denoised_image = cv2.fastNlMeansDenoising(binary_image, None, 30, 7, 21)

    return denoised_image

def getSingleCallNum(image_path):
    """
    识别并获取单本书的索书号（多行文本，包括符号“=-:./”）
    """
    # 读取分割后的索书号区域图像
    image = cv2.imread(image_path)

    # 对图像进行预处理
    preprocessed_image = preprocess_image(image)
    print("pre_image",preprocessed_image)
    base_64 = image_to_base64(image_path)
    print("base64:", base_64)

    # 使用Tesseract进行字符识别，配置为多行文本识别模式
    config = '--psm 6'  # 允许文本有多个段落
    recognized_text = pytesseract.image_to_string(gray, config=config)

    # 包含大写字母、数字和指定符号的有效字符集合
    valid_chars = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789=-:./")

    # 初始化空列表存储每行的有效索书号字符
    call_number_lines = []

    # 按行分割识别到的文本，并逐行处理
    for line in recognized_text.splitlines():
        # 筛选出每行中的有效字符
        filtered_line = ''.join([char for char in line if char in valid_chars])
        if filtered_line:
            call_number_lines.append(filtered_line)

    # 返回一个包含所有行有效字符的列表
    return call_number_lines

if __name__ == '__main__':
    image_path = './shuji.jpg'
    call_Num = hsv_get(image_path)
    print("识别的索书号：", call_Num)

In [ ]:
import base64
from dao.BaseDao import BaseDao
import cv2

class BookDao(BaseDao):
    """
    职位数据管理数据库操作类
    DAO：database access object
    """
    # 获取book表中所有记录
    def getBooks(self):
        sql = 'select * from book'
        result = self.execute(sql=sql)
        resultSet = self.fetchall()
        return resultSet

def findLabelFromName(resultSet, character):
    """
    根据字符在结果集中查找标签匹配度超过75%的记录
    返回num_info；否则返回-1
    """
    for item in resultSet:
        count = 0
        length = len(character)
        for i in character:
            if i in item['label']:
                count += 1
            if count / length > 0.75:
                return item['num_info']
    return -1

def save_base64_image(data, file_path):
    """
    将Base64编码的图像数据保存到指定文件路径
    """
    try:
        # 提取并解码Base64数据
        base64_data = data
        binary_data = base64.b64decode(base64_data)
        # 将二进制数据写入文件
        with open(file_path, 'wb') as f:
            f.write(binary_data)
        print("图片保存成功")
    except Exception as e:
        print(f"图片保存失败: {e}")


def image_to_base64(image_path):
    """将图像文件转换为Base64编码字符串"""
    with open(image_path, "rb") as image_file:
        # 读取图片文件内容
        image_data = image_file.read()
        # 将图片内容编码为 base64 格式
        base64_encoded = base64.b64encode(image_data)
        # 将 bytes 类型转换为字符串类型
        base64_encoded_str = base64_encoded.decode('utf-8')
        return base64_encoded_str

def seq_to_filepath(filename, id, result_dir):
    """
    根据文件名、序号及结果目录生成完整文件路径
    如果id为0，则生成的文件路径为result_dir加上filename以及.jpg扩展名。
    否则，生成的文件路径为result_dir加上filename、(id+1)（表示序列编号）以及.jpg扩展名。
    """
    if id == 0:
        filepath = result_dir + str(filename) + '.jpg'
    else:
        filepath = result_dir + str(filename) + str(id + 1) + '.jpg'
    return filepath



def draw_bounding_box(image_path, errors_box, question_box, output_path):
    """在图像上绘制边界框并保存"""
    # 读取图像
    image = cv2.imread(image_path)
    # 绘制错误框（红色，厚度为20）
    for err in errors_box:
        x_center, y_center, width, height = err
        # 计算方框的左上角和右下角坐标
        x1 = int(x_center - width / 2)
        y1 = int(y_center - height / 2)
        x2 = int(x_center + width / 2)
        y2 = int(y_center + height / 2)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), thickness=20)

    # 绘制问题框（绿色，厚度为10）
    for question in question_box:
        x_center, y_center, width, height = question
        # 计算方框的左上角和右下角坐标
        x1 = int(x_center - width / 2)
        y1 = int(y_center - height / 2)
        x2 = int(x_center + width / 2)
        y2 = int(y_center + height / 2)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), thickness=10)

    cv2.imwrite(output_path, image)


In [ ]:
print(image_to_base64('shuji.jpg'))

In [ ]:
import cv2
import numpy as np
import base64

from utils.ImageExecute import image_to_base64
from utils.OCR import send_post_request
# from utils.OCR import send_post_request
# from utils.ImageExecute import image_to_base64

def hsv_get(image_path):
    """
    再次分割书脊获得书标区域
    """
    # 读取图像
    image = cv2.imread(image_path)

    # 将图像转换为HSV颜色空间
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # 定义红色的HSV范围
    lower_red = np.array([0, 100, 100])    # 红色的低阈值
    upper_red = np.array([10, 255, 255])   # 红色的高阈值

    # 创建一个mask，其中红色区域为白色，其他区域为黑色
    mask = cv2.inRange(hsv, lower_red, upper_red)

    # 寻找红线区域的轮廓
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 设置上下偏移量
    y_offset_top = -22
    y_offset_bottom = 3

    # 在原始图像上绘制红线区域的轮廓（仅作为示例）
    if contours:
        # 对轮廓按面积排序，取最大的两个轮廓
        contours = sorted(contours, key=cv2.contourArea, reverse=True)[:2]

        # 获取两条红线的 y 坐标
        y_coords = []
        for contour in contours:
            _, y, _, _ = cv2.boundingRect(contour)
            y_coords.append(y)

        # 确定上下两条红线的 y 坐标并应用偏移量
        y_coords.sort()
        y_top = max(y_coords[0] - y_offset_top, 0)
        y_bottom = min(y_coords[1] + y_offset_bottom, image.shape[0])

        # 提取两条红线之间的区域
        between_region = image[y_top:y_bottom, :]
        return between_region
    else:
        print("未找到红线区域，请调整阈值或检查图像")
        return 0
    
def preprocess_image(image):
    """
    对图像进行预处理，包括对比度增强、二值化和去噪。
    """
    # 增强对比度
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    image = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    # 转为灰度图
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 二值化
    _, binary_image = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 去噪
    denoised_image = cv2.fastNlMeansDenoising(binary_image, None, 30, 7, 21)

    return denoised_image

def image2base64(image):
    """
    将预处理后的图像转换为Base64编码。
    """
    # 将图像编码为JPEG格式
    ret, buffer = cv2.imencode('.jpg', image)
    if not ret:
        raise ValueError("Failed to encode the image.")
    
    # 将缓冲区转换为Base64编码
    image_base64 = base64.b64encode(buffer).decode('utf-8')
    
    return image_base64


In [50]:
def getSingleCallNum(image_path):
    """
    识别并获取单本书的索书号（多行文本，包括符号“=-:./”）
    """
    # 读取分割后的索书号区域图像
    image = hsv_get(image_path)

    # 对图像进行预处理
    preprocessed_image = preprocess_image(image)
    
    base_64 = image2base64(preprocessed_image)

    repons = send_post_request(base_64).json()
    recognized_text = []
    for locate, element, conf in repons['data']['raw_out']:
        print("locate:", locate)
        print("element:", element)
        print("conf:", conf)
        for item in element:
            characters = list(item)
            recognized_text.extend(characters)
    print(recognized_text)
    
    # 包含大写字母、数字和指定符号的有效字符集合
    valid_chars = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789=-:./")

    

    current_ascii = [ord(c) for c in recognized_text]
    print(current_ascii)
    # 返回一个包含所有行有效字符的列表
    return call_number_lines

In [51]:
getSingleCallNum("shuji.jpg")

locate: [43.0, 23.0, 44.0, 19.14285659790039, -0.0]
element: 1267
conf: 0.9002300798892975
locate: [42.0, 46.0, 56.0, 19.14285659790039, 0.0]
element: 585=4
conf: 0.9997530102729797
['1', '2', '6', '7', '5', '8', '5', '=', '4']
[49, 50, 54, 55, 53, 56, 53, 61, 52]


AttributeError: 'list' object has no attribute 'splitlines'

In [43]:
base_64 = image_to_base64('shuji.jpg')
repons = send_post_request(base_64).json()
print("打印的值:", repons['data']['raw_out'])

打印的值: [[[5.0, 242.0, 6.0, 10.571428298950195, -0.0], '', 0.0], [[49.5, 247.5, 19.0, 20.571428298950195, -0.0], '伯', 0.8116766810417175], [[89.0, 247.5, 8.0, 9.14285659790039, 0.0], '', 0.0], [[50.0, 259.5, 20.0, 17.71428680419922, -0.0], '杨', 0.9998886585235596], [[67.0, 259.0, 8.0, 13.428571701049805, 0.0], '', 0.0], [[85.5, 257.5, 19.0, 17.71428680419922, 0.0], '', 0.0], [[48.5, 274.5, 19.0, 17.71428680419922, -0.0], '杂', 0.9993546605110168], [[48.5, 288.0, 19.0, 16.28571319580078, -0.0], '文', 0.9995074272155762], [[78.5, 284.5, 33.0, 32.0, -0.0], '理', 0.9890865087509155], [[48.5, 301.5, 19.0, 17.71428680419922, -0.0], '精', 0.9998095631599426], [[47.0, 316.5, 18.0, 17.71428680419922, 0.0], '选', 0.9815452694892883], [[47.5, 337.5, 19.0, 17.71428680419922, -0.0], '女', 0.9999971389770508], [[47.0, 352.5, 20.0, 17.71428680419922, -0.0], '性', 0.9994524121284485], [[78.0, 346.0, 34.0, 33.42857360839844, -0.0], '千', 0.9947946667671204], [[1.0999975204467773, 375.1999816894531, 6.68328142166